# Notebook 03: Calibration Analysis

**Question:** When the optimised model says 'I'm 90% confident', is it actually right 90% of the time? Does quantisation change this?

**Method:** Bin predictions by confidence score. For each bin, compare mean confidence against actual accuracy. A perfectly calibrated model sits on the diagonal. The Expected Calibration Error (ECE) summarises the gap.

**Variants:** full, onnx_fp32, onnx_int8

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from quantscope import load_predictions, compute_calibration, print_calibration_report

sns.set_theme(style='whitegrid')

In [ ]:
import os
os.makedirs('../results/charts', exist_ok=True)
print('✅ charts/ folder created')

In [ ]:
data   = load_predictions('../results/all_predictions.json')
labels = data['labels']

print(f'Test set: {len(labels)} samples')

In [ ]:
# Compute calibration for all three variants
cal = {}
for variant in ['full', 'onnx_fp32', 'onnx_int8']:
    cal[variant] = compute_calibration(
        labels,
        data[variant]['predictions'],
        data[variant]['confidences'],
    )
    print_calibration_report(variant, cal[variant])
    print()

In [ ]:
# Chart 2: Calibration curves
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Calibration Curves: Confidence vs Actual Accuracy', fontsize=13)

colors = {'full': '#3b82f6', 'onnx_fp32': '#f59e0b', 'onnx_int8': '#ef4444'}

for ax, variant in zip(axes, ['full', 'onnx_fp32', 'onnx_int8']):
    bins      = cal[variant]['bins']
    conf_vals = [b['bin_conf'] for b in bins]
    acc_vals  = [b['bin_acc']  for b in bins]
    ece       = cal[variant]['ece']

    # Perfect calibration diagonal
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration', alpha=0.5)

    # Model calibration curve
    ax.plot(conf_vals, acc_vals, 'o-', color=colors[variant],
            linewidth=2, markersize=6, label=f'{variant} (ECE={ece:.4f})')

    # Fill gap area
    ax.fill_between(conf_vals, conf_vals, acc_vals,
                    alpha=0.15, color=colors[variant])

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Mean Confidence')
    ax.set_ylabel('Actual Accuracy')
    ax.set_title(variant)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../results/charts/calibration_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved calibration_curves.png')

In [ ]:
# ECE comparison summary
print('ECE Summary (lower = better calibrated):')
print('-' * 35)
for variant in ['full', 'onnx_fp32', 'onnx_int8']:
    ece = cal[variant]['ece']
    print(f'  {variant:<12} : {ece:.4f}')

# Confidence distribution comparison
print('\nMean confidence scores:')
print('-' * 35)
for variant in ['full', 'onnx_fp32', 'onnx_int8']:
    confs = data[variant]['confidences']
    print(f'  {variant:<12} : {np.mean(confs):.4f} (std={np.std(confs):.4f})')

In [ ]:
# Save calibration results
cal_summary = {
    variant: {
        'ece' : cal[variant]['ece'],
        'bins': cal[variant]['bins'],
    }
    for variant in cal
}

with open('../results/calibration_summary.json', 'w') as f:
    json.dump(cal_summary, f, indent=2)

print('✅ Saved calibration_summary.json')